# 基础的invoke调用就不说了，内部根据对象的不同，写法也有一定的差异，常见的就是字典列表

In [ ]:
#调用模型
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 定义LLM模型
model = ChatOpenAI(model="gpt-4o-mini", temperature=0, timeout=10,max_tokens=1000)

## 1、流式输出

##### 流式输出就是用deepseek、chatgpt那种逐步显示输出，而invoke就是一下子把输出全部给到你，流式输出大概是为了适应人类阅读习惯

##### 与 invoke（） 相反，invoke（）在模型生成完完整响应后返回单个 AIMessage，stream()返回多个 AIMessageChunk 对象，每个对象包含输出文本的一部分。重要的是，流中的每个块都设计为通过求和收集到完整消息中：stream()

In [4]:
full = None  # None | AIMessageChunk
for chunk in model.stream("What color is the sky? just tell me answer"):
    full = chunk if full is None else full + chunk
    print(full.text)

print(full.content_blocks)
# [{"type": "text", "text": "The sky is typically blue..."}]


The
The sky
The sky is
The sky is typically
The sky is typically blue
The sky is typically blue.
The sky is typically blue.
The sky is typically blue.
The sky is typically blue.
[{'type': 'text', 'text': 'The sky is typically blue.'}]


## 2、批次输出

##### 对模型批处理一组独立请求可以显着提高性能并降低成本，因为处理可以并行完成

In [5]:
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])
for response in responses:
    print(response)

content='Parrots have colorful feathers for several reasons, primarily related to their survival, communication, and mating. Here are some key factors:\n\n1. **Camouflage**: In their natural habitats, which often include lush forests and tropical environments, bright colors can help parrots blend in with the vibrant foliage and flowers. This camouflage can protect them from predators.\n\n2. **Communication**: Colorful feathers can play a role in social interactions among parrots. Bright colors can signal health and vitality, helping to establish dominance or attract mates. Parrots often use their plumage in displays to communicate with each other.\n\n3. **Sexual Selection**: Many species of parrots exhibit sexual dimorphism, where males and females have different coloration. Brightly colored males may attract females, as vibrant colors can indicate good genes and overall fitness. This can lead to a preference for mates with more vivid plumage.\n\n4. **Thermoregulation**: In some cases,

##### 默认情况下，batch（） 将仅返回整个批次的最终输出。如果要在完成生成时接收每个单独输入的输出，可以使用 batch_as_completed（） 流式传输结果

In [ ]:
for response in model.batch_as_completed([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
]):
    print(response)

##### 使用 batch_as_completed（） 时，结果可能会无序出现。每个都包括用于匹配的输入索引，以根据需要重建原始订单。

##### 使用 batch（） 或 batch_as_completed（） 处理大量输入时，您可能希望控制并行调用的最大数量。这可以通过在 RunnableConfig 字典中设置 max_concurrency 属性来完成。

In [ ]:
model.batch(
    [
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
],config={
        'max_concurrency': 5,  # Limit to 5 parallel calls
    }
)

## 3、模型调用工具（注意是模型而不是agent）

##### 要使已定义的工具可供模型使用，必须使用 bind_tools（） 绑定它们。在后续调用中，模型可以根据需要选择调用任何绑定工具。
##### 一些模型提供程序提供了可以通过模型或调用参数（例如 ChatOpenAI、ChatAnthropic）启用的内置工具。有关详细信息，请查看相应的提供商参考。比如ChatOpenAI，它可以利用convert_to_openai_function()方法将工具转换为函数，然后用invoke传给模型，也能实现工具调用。

In [6]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."


model_with_tools = model.bind_tools([get_weather])       #主要是bind_tools方法

response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


## 4、结构化输出

##### 可以要求模型以与给定模式匹配的格式提供响应。这对于确保输出可以轻松解析并在后续处理中使用非常有用。LangChain 支持多种模式类型和方法来强制执行结构化输出。

### （1）Pydantic
##### Pydantic 模型提供了最丰富的特征集，包括字段验证、描述和嵌套结构。

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="这部电影的名称")
    year: int = Field(..., description="这部电影上映的年份")
    director: str = Field(..., description="这部电影的导演")
    rating: float = Field(..., description="这部电影的评分，以10分为满分")

model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("提供关于电影《盗梦空间》的细节")
print(response)  # Movie(title="Inception", year=2010, director="Christopher Nolan", rating=8.8)

title='盗梦空间' year=2010 director='克里斯托弗·诺兰' rating=8.8


### （2）TypedDict
##### TypedDict使用 Python 的内置类型提供了一个更简单的替代方案，非常适合不需要运行时验证的情况。

In [8]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "这部电影的名称"]
    year: Annotated[int, ..., "这部电影上映的年份"]
    director: Annotated[str, ..., "这部电影的导演"]
    rating: Annotated[float, ..., "这部电影的评分，以10分为满分"]

model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("提供关于电影《盗梦空间》的细节")
print(response)  # {'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}

{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 8.8}


### （3）JSON架构
##### 为了获得最大程度的控制或互作性，可以提供原始 JSON Schema

In [9]:
import json

json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "这部电影的名称"
        },
        "year": {
            "type": "integer",
            "description": "这部电影上映的年份"
        },
        "director": {
            "type": "string",
            "description": "这部电影的导演"
        },
        "rating": {
            "type": "number",
            "description": "这部电影的评分，以10分为满分"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

model_with_structure = model.with_structured_output(
    json_schema,
    method="json_schema",
)
response = model_with_structure.invoke("提供关于电影《盗梦空间》的细节")
print(response)  # {'title': 'Inception', 'year': 2010, ...}

{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 8.8}


### 结构化输出的关键考虑因素：
#### （1）方法参数：某些提供程序支持不同的方法 （'json_schema'，'function_calling'，'json_mode')
##### &#160;&#160;&#160;&#160;&#160;&#160;&#160;&#160;'json_schema'通常是指提供商提供的专用结构化输出功能
##### &#160;&#160;&#160;&#160;&#160;&#160;&#160;&#160;'function_calling'通过强制按照给定模式进行工具调用来派生结构化输出
##### &#160;&#160;&#160;&#160;&#160;&#160;&#160;&#160;'json_mode'是某些提供商提供的前身 - 它生成有效的 json，但必须在提示中描述模式'json_schema'
#### （2）包括原始：用于获取解析的输出和原始 AI 消息include_raw=True
#### （3）验证：Pydantic 模型提供自动验证，而 JSON Schema 需要手动验证TypedDict

#### include_raw=True

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="这部电影的名称")
    year: int = Field(..., description="这部电影上映的年份")
    director: str = Field(..., description="这部电影的导演")
    rating: float = Field(..., description="这部电影的评分，以10分为满分")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("提供关于电影《盗梦空间》的细节")
print(response)  # Movie(title="Inception", year=2010, director="Christopher Nolan", rating=8.8)

{'raw': AIMessage(content='{"title":"盗梦空间","year":2010,"director":"克里斯托弗·诺兰","rating":8.8}', additional_kwargs={'parsed': Movie(title='盗梦空间', year=2010, director='克里斯托弗·诺兰', rating=8.8), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 240, 'total_tokens': 270, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CVcwOu0eXjM9acFWybjlLasyaBF0W', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--4d82a0db-0171-46d4-a81c-c0a517cfbf9d-0', usage_metadata={'input_tokens': 240, 'output_tokens': 30, 'total_tokens': 270, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), 'parsed': Movie(title='盗梦空间', year=2010, dire

### 模式可以嵌套

In [12]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("提供关于电影《盗梦空间》的细节")
print(response)

title='盗梦空间' year=2010 cast=[Actor(name='莱昂纳多·迪卡普里奥', role='多米尼克·柯布'), Actor(name='约瑟夫·高登-莱维特', role='阿瑟'), Actor(name='艾伦·佩吉', role='阿里阿德涅'), Actor(name='汤姆·哈迪', role='艾姆斯'), Actor(name='肯·渡边', role='斋藤'), Actor(name='玛丽昂·歌迪亚', role='梅尔')] genres=['科幻', '动作', '冒险'] budget=160.0
